In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import xarray as xr
import numpy as np
from distributed import Client, LocalCluster
import dask
import pickle
import os
from scipy.stats import linregress
from matplotlib.lines import Line2D
import seaborn as sns
from scipy.stats import linregress
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
import joblib # For saving our models
import time
import emcee
from matplotlib.ticker import FuncFormatter
plt.rcParams['text.usetex'] = True
import corner # The library for creating corner plots

# Read in pickle files

In [2]:
#======================
# Load CNTL Dictionaryp
#======================
read_path = '/glade/u/home/mckenna/scratch/ppe_processed_files/'
dict_list = sorted(glob.glob(read_path + '*.pkl'))

cntl_path_id = [i for i, f in enumerate(dict_list) if '_00' in f]
cntl_dict_path = dict_list[cntl_path_id[0]]

with open(cntl_dict_path, 'rb') as f:
    cntl_dict = pickle.load(f)

cntl_pd_dict = cntl_dict['pD']

In [3]:
#dict_list

In [20]:
def area_weighted_mean(data, area, mask=None):
    # Ensure area is a 1D array
    area = np.asarray(area)

    if mask is not None:
        # If mask is 2D and data is 1D (e.g., time-averaged), reduce the mask along time axis
        if mask.ndim == 2 and data.ndim == 1:
            mask = np.any(mask, axis=0)

        # Now apply the mask
        data = np.where(mask, data, np.nan)

    valid = ~np.isnan(data)
    return np.nansum(data[valid] * area[valid]) / np.nansum(area[valid])

    
def compute_warm_stratiform_mask(case_dict):
    # --- 1. Define Instantaneous Masks ---
    iwp = case_dict['iwp']
    icc = case_dict['icc']
    ttop = case_dict['ttop']
    lcc = case_dict['lcc']
    ocnfrac = case_dict['ocnfrac']
    omega_500 = case_dict['omega_500']
    omega_700 = case_dict['omega_700']
    th7001000 = case_dict['th7001000']
    
    warm_mask = ((iwp < 1e-3) & (icc < 1e-12) & (ttop > 273.15) & (lcc > 0.001) & (ocnfrac > 0.99))
    warm_overcast_mask = (warm_mask & (lcc > 0.9) & (ocnfrac > 0.99))
    strat_mask_ms = ((omega_500 > 10.) & (omega_700 > 10.) & (th7001000 > 18.55) & (ocnfrac > 0.99))

    # --- 2. Calculate Climatological Fractions (for visualization and regime definition) ---
    warm_frac = warm_mask.mean(axis=0)
    warm_overcast_frac = warm_overcast_mask.mean(axis=0)
    strat_frac = strat_mask_ms.mean(axis=0)

    # --- 3. Define the Final 'warm_strat' Regime Mask (for analysis) ---
    strat_column_mask = strat_frac > 0.3
    strat_column_mask_broadcast = np.broadcast_to(strat_column_mask, warm_mask.shape)
    warm_strat_mask = warm_mask & strat_column_mask_broadcast

    # --- 4. Calculate the 'warm_strat' Fraction (for visualization) ---
    warm_strat_frac = np.where(strat_column_mask, warm_frac, np.nan)

    # --- 5. Create Masked Arrays for Plotting ---
    # Using a small positive threshold like 0.01 is often safer than 0.0
    plot_thresh = 0.1 
    warm_masked = np.ma.masked_where(warm_frac < plot_thresh, warm_frac)
    warm_overcast_masked = np.ma.masked_where(warm_overcast_frac < plot_thresh, warm_overcast_frac)
    strat_masked = np.ma.masked_where(strat_frac < plot_thresh, strat_frac)
    warm_strat_masked = np.ma.masked_where(np.isnan(warm_strat_frac) | (warm_strat_frac < plot_thresh), warm_strat_frac)

    # --- 6. Return all necessary products ---
    return {
        # The crucial analysis masks
        'warm_mask': warm_mask,
        'warm_overcast_mask': warm_overcast_mask,
        'strat_mask': strat_mask_ms,
        'warm_strat_mask': warm_strat_mask, # The primary output for compute_metrics
        
        # The fractions for visualization
        'warm_frac': warm_frac,
        'warm_overcast_frac': warm_overcast_frac,
        'strat_frac': strat_frac,
        'warm_strat_frac': warm_strat_frac,
        
        # The plottable masked arrays
        'warm_masked': warm_masked,
        'warm_overcast_masked': warm_overcast_masked,
        'strat_masked': strat_masked,
        'warm_strat_masked': warm_strat_masked,
    }
    
def compute_metrics(case_dict, label='warm',aero='PD'):
    # Extract primary data
    area = case_dict['area']
    lcc = case_dict['lcc']
    lwp = case_dict['lwp']
    cdnc = case_dict['cdnc']
    reff = case_dict['cdr']
    prect = case_dict['prect']
    autoconv = case_dict['autoconv']
    accretn = case_dict['accretn']
    precl = case_dict['precl']
    time_mask = case_dict[f'{label}_mask'].astype(bool)

    # Masking: Compute valid_lcc_mask upfront based on label
    valid_lcc_mask = ((lcc > 0.1) & time_mask) if label == 'warm' else ((lcc > 0.9) & time_mask)

    # Define helper function for masked ratio mean (for in-cloud values)
    def masked_ratio_mean(var):
        ratio = np.where(valid_lcc_mask, var / lcc, np.nan)  # Normalize by LCC
        return area_weighted_mean(np.nanmean(ratio, axis=0), area, mask=np.any(valid_lcc_mask, axis=0))

    # In-cloud metrics (normalize by LCC)
    out = {
        f'in_cloud_lwp_{label}': masked_ratio_mean(lwp),
        f'in_cloud_cdnc_{label}': masked_ratio_mean(cdnc),
        f'in_cloud_reff_{label}': masked_ratio_mean(reff),
    }

    # Mask variables globally (apply valid_lcc_mask directly)
    autoconv = np.where(valid_lcc_mask, autoconv, np.nan)
    accretn = np.where(valid_lcc_mask, accretn, np.nan)
    precl = np.where(valid_lcc_mask, precl, np.nan)
    prect = np.where(valid_lcc_mask, prect, np.nan)
    lcc = np.where(valid_lcc_mask, lcc, np.nan) 

    # Area-weighted global means
    out.update({
        f'lcc_{label}': area_weighted_mean(np.nanmean(lcc, axis=0), area, mask=np.any(valid_lcc_mask, axis=0)),
        f'autoconv_{label}': area_weighted_mean(np.nanmean(autoconv, axis=0), area, mask=np.any(valid_lcc_mask, axis=0)),
        f'accretn_{label}': area_weighted_mean(np.nanmean(accretn, axis=0), area, mask=np.any(valid_lcc_mask, axis=0)),
        f'precl_{label}': area_weighted_mean(np.nanmean(precl, axis=0), area, mask=np.any(valid_lcc_mask, axis=0)),
        f'prect_{label}': area_weighted_mean(np.nanmean(prect, axis=0), area, mask=np.any(valid_lcc_mask, axis=0)),
    })

    # Compute global LCC average for the regime (temporal and spatial weighted fraction)
    masked_lcc = np.where(time_mask, lcc, 0.0)  # Set LCC to 0 for points outside regime mask
    global_lcc = area_weighted_mean(np.nanmean(masked_lcc, axis=0), area)  # Temporal mean first, then spatially weighted
    out[f'global_lcc_{label}'] = global_lcc
    #print(label,':',global_lcc)

    # Warm rain occurrence frequency (under cloudy conditions)
    precip_thresh = 1.e-3  # mm/hr
    valid_precip_mask = time_mask & (prect > precip_thresh)
    out[f'warm_rain_freq_{label}'] = np.sum(valid_precip_mask * area) / np.sum(time_mask * area)

    # Precipitation amplification frequency (under precipitating conditions)
    # This now answers: "Of the area that is already precipitating, what fraction is precipitating *strongly*?"
    precip_amp_thresh = 0.1 # mm/hr
    valid_amplified_mask = time_mask & (prect > precip_amp_thresh)

    # The numerator is the area of strong precipitation (your original `valid_amplified_mask`)
    numerator_paf = np.sum(valid_amplified_mask * area)

    # The denominator is the total area that is precipitating at the lower threshold
    # (the numerator from your WROF calculation). This is the key change.
    denominator_paf = np.sum(valid_precip_mask * area)

    # Avoid division by zero if there are no precipitating points at all
    if denominator_paf > 0:
        out[f'precip_amp_freq_{label}'] = numerator_paf / denominator_paf
    else:
        # If there are no raining points, there can be no amplification.
        out[f'precip_amp_freq_{label}'] = 0.0

    print('')
    print('Label:',label+','+aero)
    print('WROF:',out[f'warm_rain_freq_{label}'])
    print('PAF:',out[f'precip_amp_freq_{label}'])

    return out



def compute_adjustments(pd_dict, pi_dict, cntl_pd_dict,mask=None):
    out = {}

    if mask == None:
        mask_pd = None
        mask_pi = None
        mask_cntl_pd = None
    elif mask == 'warm':
        mask_pd = pd_dict['warm_mask'].astype(bool)
        mask_pi = pd_dict['warm_mask'].astype(bool)
        mask_cntl_pd = cntl_pd_dict['warm_mask'].astype(bool)
    elif mask == 'warm_overcast':
        mask_pd = pd_dict['warm_overcast_mask'].astype(bool)
        mask_pi = pd_dict['warm_overcast_mask'].astype(bool)
        mask_cntl_pd = cntl_pd_dict['warm_overcast_mask'].astype(bool)
    elif mask == 'warm_strat':
        mask_pd = pd_dict['warm_strat_mask'].astype(bool)
        mask_pi = pd_dict['warm_strat_mask'].astype(bool)
        mask_cntl_pd = cntl_pd_dict['warm_strat_mask'].astype(bool)

    if mask == None:
        # LWP Adjustment
        lwp_pd = area_weighted_mean(np.nanmean(pd_dict['lwp'], axis=0), pd_dict['area'], mask_pd)
        lwp_pi = area_weighted_mean(np.nanmean(pi_dict['lwp'], axis=0), pi_dict['area'], mask_pi)
        lwp_cntl = area_weighted_mean(np.nanmean(cntl_pd_dict['lwp'], axis=0), cntl_pd_dict['area'], mask_cntl_pd)
        out['lwp_adj'] = np.log(lwp_pd / lwp_pi)
        out['lwp_delta'] = lwp_pd - lwp_pi
        out['lwp_bias'] = lwp_pd - lwp_cntl
    
        # CDNC Adjustment
        cdnc_pd = area_weighted_mean(np.nanmean(pd_dict['cdnc'], axis=0), pd_dict['area'], mask_pd)
        cdnc_pi = area_weighted_mean(np.nanmean(pi_dict['cdnc'], axis=0), pi_dict['area'], mask_pi)
        cdnc_cntl = area_weighted_mean(np.nanmean(cntl_pd_dict['cdnc'], axis=0), cntl_pd_dict['area'], mask_cntl_pd)
        out['cdnc_adj'] = np.log(cdnc_pd / cdnc_pi)
        out['cdnc_delta'] = cdnc_pd - cdnc_pi
        out['cdnc_bias'] = cdnc_pd - cdnc_cntl

        # Reff Adjustment
        reff_pd = area_weighted_mean(np.nanmean(pd_dict['cdr'], axis=0), pd_dict['area'], mask_pd) 
        reff_pi = area_weighted_mean(np.nanmean(pi_dict['cdr'], axis=0), pi_dict['area'], mask_pi) 
        reff_cntl = area_weighted_mean(np.nanmean(cntl_pd_dict['cdr'], axis=0), cntl_pd_dict['area'], mask_cntl_pd) 
        out['reff_adj'] = np.log(reff_pd / reff_pi)
        out['reff_delta'] = reff_pd - reff_pi
        out['reff_bias'] = reff_pd - reff_cntl
    
        # LCC adjustment
        lcc_pd = area_weighted_mean(np.nanmean(pd_dict['lcc'], axis=0), pd_dict['area'], mask_pd)
        lcc_pi = area_weighted_mean(np.nanmean(pi_dict['lcc'], axis=0), pi_dict['area'], mask_pi)
        lcc_cntl = area_weighted_mean(np.nanmean(cntl_pd_dict['lcc'], axis=0), cntl_pd_dict['area'], mask_cntl_pd)
        out['lcc_adj'] = np.log(lcc_pd / lcc_pi)
        out['lcc_delta'] = lcc_pd - lcc_pi
        out['lcc_bias'] = lcc_pd - lcc_cntl

        # AUTOCONV
        autoconv_pd = area_weighted_mean(np.nanmean(pd_dict['autoconv'], axis=0), pd_dict['area'], mask_pd)
        autoconv_pi = area_weighted_mean(np.nanmean(pi_dict['autoconv'], axis=0), pi_dict['area'], mask_pi)
        autoconv_cntl = area_weighted_mean(np.nanmean(cntl_pd_dict['autoconv'], axis=0), cntl_pd_dict['area'], mask_cntl_pd)
        out['autoconv_adj'] = np.log(autoconv_pd / autoconv_pi)
        out['autoconv_delta'] = autoconv_pd - autoconv_pi
        out['autoconv_bias'] = autoconv_pd - autoconv_cntl
        
        # ACCRETN
        accretn_pd = area_weighted_mean(np.nanmean(pd_dict['accretn'], axis=0), pd_dict['area'], mask_pd)
        accretn_pi = area_weighted_mean(np.nanmean(pi_dict['accretn'], axis=0), pi_dict['area'], mask_pi)
        accretn_cntl = area_weighted_mean(np.nanmean(cntl_pd_dict['accretn'], axis=0), cntl_pd_dict['area'], mask_cntl_pd)
        out['accretn_adj'] = np.log(accretn_pd / accretn_pi)
        out['accretn_delta'] = accretn_pd - accretn_pi
        out['accretn_bias'] = accretn_pd - accretn_cntl

        # PRECT
        prect_pd = area_weighted_mean(np.nanmean(pd_dict['prect'], axis=0), pd_dict['area'], mask_pd)
        prect_pi = area_weighted_mean(np.nanmean(pi_dict['prect'], axis=0), pi_dict['area'], mask_pi)
        prect_cntl = area_weighted_mean(np.nanmean(cntl_pd_dict['prect'], axis=0), cntl_pd_dict['area'], mask_cntl_pd)
        out['prect_adj'] = np.log(prect_pd / prect_pi)
        out['prect_delta'] = prect_pd - prect_pi
        out['prect_bias'] = prect_pd - prect_cntl

        # PRECL
        precl_pd = area_weighted_mean(np.nanmean(pd_dict['precl'], axis=0), pd_dict['area'], mask_pd)
        precl_pi = area_weighted_mean(np.nanmean(pi_dict['precl'], axis=0), pi_dict['area'], mask_pi)
        precl_cntl = area_weighted_mean(np.nanmean(cntl_pd_dict['precl'], axis=0), cntl_pd_dict['area'], mask_cntl_pd)
        out['precl_adj'] = np.log(precl_pd / precl_pi)
        out['precl_delta'] = precl_pd - precl_pi
        out['precl_bias'] = precl_pd - precl_cntl

    elif mask in ['warm', 'warm_overcast', 'warm_strat']:

        # Define a small epsilon
        epsilon = 1e-12
        
        # LWP
        lwp_pd = pd_dict[f'in_cloud_lwp_{mask}']
        lwp_pi = pi_dict[f'in_cloud_lwp_{mask}']
        lwp_cntl = cntl_pd_dict[f'in_cloud_lwp_{mask}']
        out[f'lwp_adj_{mask}'] = np.log(lwp_pd / lwp_pi)
        out[f'lwp_delta_{mask}'] = lwp_pd - lwp_pi
        out[f'lwp_bias_{mask}'] = lwp_pd - lwp_cntl

        # CDNC
        cdnc_pd = pd_dict[f'in_cloud_cdnc_{mask}']
        cdnc_pi = pi_dict[f'in_cloud_cdnc_{mask}']
        cdnc_cntl = cntl_pd_dict[f'in_cloud_cdnc_{mask}']
        out[f'cdnc_adj_{mask}'] = np.log(cdnc_pd / cdnc_pi)
        out[f'cdnc_delta_{mask}'] = cdnc_pd - cdnc_pi
        out[f'cdnc_bias_{mask}'] = cdnc_pd - cdnc_cntl

        # Reff
        reff_pd = pd_dict[f'in_cloud_reff_{mask}'] 
        reff_pi = pi_dict[f'in_cloud_reff_{mask}'] 
        reff_cntl = cntl_pd_dict[f'in_cloud_reff_{mask}']
        out[f'reff_adj_{mask}'] = np.log(reff_pd / reff_pi)
        out[f'reff_delta_{mask}'] = reff_pd - reff_pi
        out[f'reff_bias_{mask}'] = reff_pd - reff_cntl

        # WROF
        warm_rain_freq_pd = pd_dict[f'warm_rain_freq_{mask}']
        warm_rain_freq_pi = pi_dict[f'warm_rain_freq_{mask}']
        warm_rain_freq_cntl = cntl_pd_dict[f'warm_rain_freq_{mask}']
        out[f'warm_rain_freq_adj_{mask}'] = np.log((warm_rain_freq_pd + epsilon) / (warm_rain_freq_pi + epsilon))
        #out[f'warm_rain_freq_adj_{mask}'] = np.log(warm_rain_freq_pd / warm_rain_freq_pi)
        out[f'warm_rain_freq_delta_{mask}'] = warm_rain_freq_pd - warm_rain_freq_pi
        out[f'warm_rain_freq_bias_{mask}'] = warm_rain_freq_pd - warm_rain_freq_cntl

        # PAF
        precip_amp_freq_pd = pd_dict[f'precip_amp_freq_{mask}']
        precip_amp_freq_pi = pi_dict[f'precip_amp_freq_{mask}']
        precip_amp_freq_cntl = cntl_pd_dict[f'precip_amp_freq_{mask}']
        out[f'precip_amp_freq_adj_{mask}'] = np.log((precip_amp_freq_pd + epsilon) / (precip_amp_freq_pi + epsilon))
        #out[f'precip_amp_freq_adj_{mask}'] = np.log(precip_amp_freq_pd / precip_amp_freq_pi)
        out[f'precip_amp_freq_delta_{mask}'] = precip_amp_freq_pd - precip_amp_freq_pi
        out[f'precip_amp_freq_bias_{mask}'] = precip_amp_freq_pd - precip_amp_freq_cntl

        # LCC adjustment
        lcc_pd = pd_dict[f'lcc_{mask}']
        lcc_pi = pi_dict[f'lcc_{mask}']
        lcc_cntl = cntl_pd_dict[f'lcc_{mask}']
        out[f'lcc_adj_{mask}'] = np.log(lcc_pd / lcc_pi)
        out[f'lcc_delta_{mask}'] = lcc_pd - lcc_pi
        out[f'lcc_bias_{mask}'] = lcc_pd - lcc_cntl

        # Global LCC adjustment
        global_lcc_pd = pd_dict[f'global_lcc_{mask}']
        global_lcc_pi = pi_dict[f'global_lcc_{mask}']
        global_lcc_cntl = cntl_pd_dict[f'global_lcc_{mask}']
        out[f'global_lcc_adj_{mask}'] = np.log(global_lcc_pd / global_lcc_pi)
        out[f'global_lcc_delta_{mask}'] = global_lcc_pd - global_lcc_pi
        out[f'global_lcc_bias_{mask}'] = global_lcc_pd - global_lcc_cntl

        # AUTOCONV
        autoconv_pd = pd_dict[f'autoconv_{mask}']
        autoconv_pi = pi_dict[f'autoconv_{mask}']
        autoconv_cntl = cntl_pd_dict[f'autoconv_{mask}']
        out[f'autoconv_adj_{mask}'] = np.log(autoconv_pd / autoconv_pi)
        out[f'autoconv_delta_{mask}'] = autoconv_pd - autoconv_pi
        out[f'autoconv_bias_{mask}'] = autoconv_pd - autoconv_cntl
        
        # ACCRETN
        accretn_pd = pd_dict[f'accretn_{mask}']
        accretn_pi = pi_dict[f'accretn_{mask}']
        accretn_cntl = cntl_pd_dict[f'accretn_{mask}']
        out[f'accretn_adj_{mask}'] = np.log(accretn_pd / accretn_pi)
        out[f'accretn_delta_{mask}'] = accretn_pd - accretn_pi
        out[f'accretn_bias_{mask}'] = accretn_pd - accretn_cntl

        # PRECT
        prect_pd = pd_dict[f'prect_{mask}']
        prect_pi = pi_dict[f'prect_{mask}']
        prect_cntl = cntl_pd_dict[f'prect_{mask}']
        out[f'prect_adj_{mask}'] = np.log(prect_pd / prect_pi)
        out[f'prect_delta_{mask}'] = prect_pd - prect_pi
        out[f'prect_bias_{mask}'] = prect_pd - prect_cntl

        # PRECL
        precl_pd = pd_dict[f'precl_{mask}']
        precl_pi = pi_dict[f'precl_{mask}']
        precl_cntl = cntl_pd_dict[f'precl_{mask}']
        out[f'precl_adj_{mask}'] = np.log(precl_pd / precl_pi)
        out[f'precl_delta_{mask}'] = precl_pd - precl_pi
        out[f'precl_bias_{mask}'] = precl_pd - precl_cntl
        
    #=====================================================
    # ERFs & CRE
    #=====================================================
    if mask == None:
        # ----------------------------------------------------------
        # --- Calculate the RESTOM for each component and for PD and PI ---
        # ----------------------------------------------------------
        # Calculate the mean over time first (axis=0), then do the area weighting.
        # Present-Day (PD) fluxes
        restom_all_pd    = area_weighted_mean(np.nanmean(pd_dict['fsnt'] - pd_dict['flnt'], axis=0), pd_dict['area'], mask_pd)
        restom_cln_pd    = area_weighted_mean(np.nanmean(pd_dict['fsnt_d1'] - pd_dict['flnt_d1'], axis=0), pd_dict['area'], mask_pd)
        restom_clrcln_pd = area_weighted_mean(np.nanmean(pd_dict['fsntc_d1'] - pd_dict['flntc_d1'], axis=0), pd_dict['area'], mask_pd)
        restom_clr_pd    = area_weighted_mean(np.nanmean(pd_dict['fsntc'] - pd_dict['flntc'], axis=0), pd_dict['area'], mask_pd)
        
        # Pre-Industrial (PI) fluxes
        restom_all_pi    = area_weighted_mean(np.nanmean(pi_dict['fsnt'] - pi_dict['flnt'], axis=0), pi_dict['area'], mask_pi)
        restom_cln_pi    = area_weighted_mean(np.nanmean(pi_dict['fsnt_d1'] - pi_dict['flnt_d1'], axis=0), pi_dict['area'], mask_pi)
        restom_clrcln_pi = area_weighted_mean(np.nanmean(pi_dict['fsntc_d1'] - pi_dict['flntc_d1'], axis=0), pi_dict['area'], mask_pi)
        restom_clr_pi    = area_weighted_mean(np.nanmean(pi_dict['fsntc'] - pi_dict['flntc'], axis=0), pi_dict['area'], mask_pi)
        

        # ----------------------------------------------------------
        # --- Calculate the "Delta" (PD - PI) for each RESTOM component ---
        # ----------------------------------------------------------
        delta_restom_all    = restom_all_pd - restom_all_pi
        delta_restom_cln    = restom_cln_pd - restom_cln_pi
        delta_restom_clrcln = restom_clrcln_pd - restom_clrcln_pi
        
        # ----------------------------------------------------------
        # --- Calculate the final ERF components ---
        # ----------------------------------------------------------
        
        # ERF = Δ RESTOM_all
        out['erf_tot'] = delta_restom_all
        
        # ERFari = Δ RESTOM_all - Δ RESTOM_cln
        out['erf_ari'] = delta_restom_all - delta_restom_cln
        
        # ERFaci = Δ RESTOM_cln - Δ RESTOM_clrcln
        out['erf_aci'] = delta_restom_cln - delta_restom_clrcln
        
        # ERFres = Δ RESTOM_clrcln
        out['erf_res'] = delta_restom_clrcln

        out['restom_pd'] = restom_all_pd
        out['restom_pi'] = restom_all_pi
        # ----------------------------------------------------------
        # CALCULATE CLOUD RADIATIVE EFFECT (CRE) AND ITS ADJUSTMENT
        # ----------------------------------------------------------
        # CRE = All-Sky Radiation - Clear-Sky Radiation
        cre_pd = restom_all_pd - restom_clr_pd
        cre_pi = restom_all_pi - restom_clr_pi
        
        # The adjustment is the change in CRE from PI to PD
        out['cre_adj'] = cre_pd - cre_pi
        out['cre_pd'] = cre_pd
        out['cre_pi'] = cre_pi

        # ----------------------------------------------------------
        # --- Calculate CERES-like Top-of-Atmosphere Radiation ---
        # ----------------------------------------------------------
        # Outgoing Longwave Radiation (OLR)
        olr_upwell_pd = area_weighted_mean(np.nanmean(pd_dict['flut'], axis=0), pd_dict['area'])  # All-sky FLUT
        olr_net_pd = area_weighted_mean(np.nanmean(pd_dict['flnt'], axis=0), pd_dict['area'])  # All-sky FLNT
    
        # Outgoing Shortwave Radiation (OSR)
        osr_upwell_pd = area_weighted_mean(np.nanmean(pd_dict['fsutoa'], axis=0), pd_dict['area'])  # All-sky FSUTOA
        osr_net_pd = area_weighted_mean(np.nanmean(pd_dict['fsnt'], axis=0), pd_dict['area'])  # All-sky FSNT(OA)
    
        out['olr_upwell_pd'] = olr_upwell_pd  # Mimics CERES TOA LW radiation
        out['osr_upwell_pd'] = osr_upwell_pd  # Mimics CERES TOA SW radiation
        out['olr_net_pd'] = olr_net_pd  
        out['osr_net_pd'] = osr_net_pd  
        
        # ----------------------------------------------------------
        # --- Calculate CERES-like Cloud Radiative Forcing (CRF) ---
        # ----------------------------------------------------------
        # Longwave Cloud Radiative Forcing (CRF-LW)
        #crf_lw_pd = area_weighted_mean(np.nanmean(pd_dict['flut'] - pd_dict['flutc'], axis=0), pd_dict['area'])  # Δ FLUT LONGWAVE
        crf_lw_pd = area_weighted_mean(np.nanmean(pd_dict['flnt'] - pd_dict['flntc'], axis=0), pd_dict['area'])  # Δ FLNT LONGWAVE

        # Shortwave Cloud Radiative Forcing (CRF-SW)
        #crf_sw_pd = area_weighted_mean(np.nanmean(pd_dict['fsutoa'] - pd_dict['fsutoac'], axis=0), pd_dict['area'])  # Δ FSUTOA SHORTWAVE
        crf_sw_pd = area_weighted_mean(np.nanmean(pd_dict['fsnt'] - pd_dict['fsntc'], axis=0), pd_dict['area'])  # Δ FSNT SHORTWAVE
        #print('mean(crf_lw_pd)',crf_lw_pd)
        #print('mean(flnt):',np.nanmean(pd_dict['flnt']))
        #print('mean(flntc):',np.nanmean(pd_dict['flntc']))
        #print('mean(crf_sw_pd)',crf_sw_pd)
        #print('mean(fsnt):',np.nanmean(pd_dict['fsnt']))
        #print('mean(fsntc):',np.nanmean(pd_dict['fsntc']))
        #print('mean(flut):',np.nanmean(pd_dict['flut']))
        #print('mean(flutc):',np.nanmean(pd_dict['flutc']))
        #print('mean(cre_pd):',cre_pd)
        #print(aaaaaa)
        
        out['crf_lw_pd'] = crf_lw_pd  # Mimics CERES SW forcing from clouds
        out['crf_sw_pd'] = crf_sw_pd  # Mimics CERES LW forcing from clouds
        
        # --- Step 4: Verification (optional but highly recommended) ---
        # Check if the components sum to the total. They should be very close (any difference is due to floating point math).
        verification_sum = out['erf_ari'] + out['erf_aci'] + out['erf_res']
        #print(f"Total ERF: {out['erf_tot']:.4f}")
        #print(f"Sum of Components (ari+aci+res): {verification_sum:.4f}")
        #print(f"Does it match? {np.isclose(out['erf_tot'], verification_sum)}")

    return out


In [21]:
print('a')

a


In [22]:
results = {}

dumi=0
for path in dict_list:
    case_name = os.path.basename(path).replace('.pkl', '')
    print('case_name:',case_name)


    print(f'Processing: {case_name}','; % done:',(dumi+1)/len(dict_list)*100.)
    with open(path, 'rb') as f:
        case = pickle.load(f)

    auto_fac = case['auto_fac']
    accr_fac = case['accr_fac']
    print('auto_fac :',auto_fac)
    print('accr_fac :',accr_fac)
    pd_dict = case['pD']
    pi_dict = case['pI']


    #========================================
    # This block computes masks for:
    # (1) Global (averaged w/o clouds)
    # (2) Warm clouds
    # (3) Warm marine overcast clouds
    # (4) Warm marine overcast stratiform clouds, 
    # following Medeiros & Stevens (2011)
    #========================================
    pd_mask_dict = compute_warm_stratiform_mask(pd_dict)
    pi_mask_dict = compute_warm_stratiform_mask(pi_dict)
    cntl_pd_mask_dict = compute_warm_stratiform_mask(cntl_pd_dict)

    for key in pd_mask_dict.keys():
        pd_dict[key] = pd_mask_dict[key]
        pi_dict[key] = pi_mask_dict[key]
        cntl_pd_dict[key] = cntl_pd_mask_dict[key]

    #========================================
    # This block provides the following:
    # (1) Warm rain occurrence frequency
    # (2) In-cloud avg LWP
    # (3) In-cloud avg CDNC
    # (4) In-cloud avg R_eff
    # (5) LCC (only relevant for warm clouds,
    #   but workflow is easier if we just let
    #   the calculations be performed for all
    #   categories.
    # 
    # This is done for pD, pI, & CNTL for warm,
    # warm overcast, and warm overcast stratiform
    #========================================

    metrics_warm_pd_dict = compute_metrics(pd_dict,label='warm',aero='PD')
    metrics_warm_pi_dict = compute_metrics(pi_dict,label='warm',aero='PI')
    metrics_warm_cntl_pd_dict = compute_metrics(cntl_pd_dict,label='warm',aero='CNTL_PD')
    metrics_warm_overcast_pd_dict = compute_metrics(pd_dict,label='warm_overcast',aero='PD')
    metrics_warm_overcast_pi_dict = compute_metrics(pi_dict,label='warm_overcast',aero='PI')
    metrics_warm_overcast_cntl_pd_dict = compute_metrics(cntl_pd_dict,label='warm_overcast',aero='CNTL_PD')
    metrics_warm_strat_pd_dict = compute_metrics(pd_dict,label='warm_strat',aero='PD')
    metrics_warm_strat_pi_dict = compute_metrics(pi_dict,label='warm_strat',aero='PI')
    metrics_warm_strat_cntl_pd_dict = compute_metrics(cntl_pd_dict,label='warm_strat',aero='CNTL_PD')
    print('')
    
    for key in metrics_warm_pd_dict.keys():
        pd_dict[key] = metrics_warm_pd_dict[key]
        pi_dict[key] = metrics_warm_pi_dict[key]
        cntl_pd_dict[key] = metrics_warm_cntl_pd_dict[key]
    for key in metrics_warm_overcast_pd_dict.keys():
        pd_dict[key] = metrics_warm_overcast_pd_dict[key]
        pi_dict[key] = metrics_warm_overcast_pi_dict[key]
        cntl_pd_dict[key] = metrics_warm_overcast_cntl_pd_dict[key]
    for key in metrics_warm_strat_pd_dict.keys():
        pd_dict[key] = metrics_warm_strat_pd_dict[key]
        pi_dict[key] = metrics_warm_strat_pi_dict[key]
        cntl_pd_dict[key] = metrics_warm_strat_cntl_pd_dict[key]

    #========================================
    # This block calculates pD biases and adjustments
    # (as deltas & log ratios) for the following:
    # (1) LWP
    # (2) CDNC
    # (3) Reff
    # (4) LCC
    # (5) WROF
    # (6) AUTOCONV
    # (7) ACCRETN
    # (8) PRECL
    # (9) PRECT
    # (10) PAF
    #========================================
    adj_dict = compute_adjustments(pd_dict, pi_dict, cntl_pd_dict,mask=None)
    warm_adj_dict = compute_adjustments(pd_dict, pi_dict, cntl_pd_dict,mask='warm')
    warm_overcast_adj_dict = compute_adjustments(pd_dict, pi_dict, cntl_pd_dict,mask='warm_overcast')
    warm_strat_adj_dict = compute_adjustments(pd_dict, pi_dict, cntl_pd_dict,mask='warm_strat')
    
    case_results = {}

    #=============================
    # Global-level
    #=============================
    case_results['global'] = {
        'pd': {
            'lwp': area_weighted_mean(np.nanmean(pd_dict['lwp'], axis=0), pd_dict['area']),
            'cdnc': area_weighted_mean(np.nanmean(pd_dict['cdnc'], axis=0), pd_dict['area']),
            'lcc': area_weighted_mean(np.nanmean(pd_dict['lcc'], axis=0), pd_dict['area']),
            'reff': area_weighted_mean(np.nanmean(pd_dict['cdr'], axis=0), pd_dict['area']),
            'autoconv': area_weighted_mean(np.nanmean(pd_dict['autoconv'], axis=0), pd_dict['area']),
            'accretn': area_weighted_mean(np.nanmean(pd_dict['accretn'], axis=0), pd_dict['area']),
            'prect': area_weighted_mean(np.nanmean(pd_dict['prect'], axis=0), pd_dict['area']),
            'precl': area_weighted_mean(np.nanmean(pd_dict['precl'], axis=0), pd_dict['area']),
        },
        'pi': {
            'lwp': area_weighted_mean(np.nanmean(pi_dict['lwp'], axis=0), pi_dict['area']),
            'cdnc': area_weighted_mean(np.nanmean(pi_dict['cdnc'], axis=0), pi_dict['area']),
            'lcc': area_weighted_mean(np.nanmean(pi_dict['lcc'], axis=0), pi_dict['area']),
            'reff': area_weighted_mean(np.nanmean(pi_dict['cdr'], axis=0), pi_dict['area']),
            'autoconv': area_weighted_mean(np.nanmean(pi_dict['autoconv'], axis=0), pi_dict['area']),
            'precl': area_weighted_mean(np.nanmean(pi_dict['precl'], axis=0), pi_dict['area']),
            'prect': area_weighted_mean(np.nanmean(pi_dict['prect'], axis=0), pi_dict['area']),
        },
        'cntl': {
            'lwp': area_weighted_mean(np.nanmean(cntl_pd_dict['lwp'], axis=0), cntl_pd_dict['area']),
            'cdnc': area_weighted_mean(np.nanmean(cntl_pd_dict['cdnc'], axis=0), cntl_pd_dict['area']),
            'lcc': area_weighted_mean(np.nanmean(cntl_pd_dict['lcc'], axis=0), cntl_pd_dict['area']),
            'reff': area_weighted_mean(np.nanmean(cntl_pd_dict['cdr'], axis=0), cntl_pd_dict['area']),
            'autoconv': area_weighted_mean(np.nanmean(cntl_pd_dict['autoconv'], axis=0), cntl_pd_dict['area']),
            'accretn': area_weighted_mean(np.nanmean(cntl_pd_dict['accretn'], axis=0), cntl_pd_dict['area']),
            'prect': area_weighted_mean(np.nanmean(cntl_pd_dict['prect'], axis=0), cntl_pd_dict['area']),
            'precl': area_weighted_mean(np.nanmean(cntl_pd_dict['precl'], axis=0), cntl_pd_dict['area']),
        },
        'adjustments': {
            'lwp_adj': adj_dict['lwp_adj'],
            'cdnc_adj': adj_dict['cdnc_adj'],
            'lcc_adj': adj_dict['lcc_adj'],
            'reff_adj': adj_dict['reff_adj'],
            'autoconv_adj': adj_dict['autoconv_adj'],
            'accretn_adj': adj_dict['accretn_adj'],
            'prect_adj': adj_dict['prect_adj'],
            'precl_adj': adj_dict['precl_adj'],
        },
        'deltas': {
            'lwp_delta': adj_dict['lwp_delta'],
            'cdnc_delta': adj_dict['cdnc_delta'],
            'lcc_delta': adj_dict['lcc_delta'],
            'reff_delta': adj_dict['reff_delta'],
            'autoconv_delta': adj_dict['autoconv_delta'],
            'accretn_delta': adj_dict['accretn_delta'],
            'prect_delta': adj_dict['prect_delta'],
            'precl_delta': adj_dict['precl_delta'],
        },
        'biases': {
            'lwp_bias': adj_dict['lwp_bias'],
            'cdnc_bias': adj_dict['cdnc_bias'],
            'lcc_bias': adj_dict['lcc_bias'],
            'reff_bias': adj_dict['reff_bias'],
            'autoconv_bias': adj_dict['autoconv_bias'],
            'accretn_bias': adj_dict['accretn_bias'],
            'prect_bias': adj_dict['prect_bias'],
            'precl_bias': adj_dict['precl_bias'],
        },
        'erfs':{
            'erf_tot':adj_dict['erf_tot'],
            'erf_ari':adj_dict['erf_ari'],
            'erf_aci':adj_dict['erf_aci'],
            'erf_res':adj_dict['erf_res'],
            'cre_adj':adj_dict['cre_adj'],
            'cre_pd':adj_dict['cre_pd'],
            'cre_pi':adj_dict['cre_pi'],
            'olr_upwell_pd':adj_dict['olr_upwell_pd'],
            'osr_upwell_pd':adj_dict['osr_upwell_pd'],
            'olr_net_pd':adj_dict['olr_net_pd'],
            'osr_net_pd':adj_dict['osr_net_pd'],
            'crf_lw_pd':adj_dict['crf_lw_pd'],
            'crf_sw_pd':adj_dict['crf_sw_pd'],
            'restom_pd':adj_dict['restom_pd'],
            'restom_pi':adj_dict['restom_pi'],
        }
    }


    regime_adj_dict_top_nest = {}
    regime_adj_dict_top_nest['warm'] = warm_adj_dict
    regime_adj_dict_top_nest['warm_overcast'] = warm_overcast_adj_dict
    regime_adj_dict_top_nest['warm_strat'] = warm_strat_adj_dict
    
    for mask in ['warm', 'warm_overcast', 'warm_strat']:

        regime_adj_dict = regime_adj_dict_top_nest[f'{mask}']
        
        case_results[mask] = {
        'pd': {
            'in_cloud_lwp': pd_dict[f'in_cloud_lwp_{mask}'],
            'in_cloud_cdnc': pd_dict[f'in_cloud_cdnc_{mask}'],
            'in_cloud_reff': pd_dict[f'in_cloud_reff_{mask}'],
            'warm_rain_freq': pd_dict[f'warm_rain_freq_{mask}'],
            'precip_amp_freq': pd_dict[f'precip_amp_freq_{mask}'],
            'lcc': pd_dict[f'lcc_{mask}'],
            'global_lcc': pd_dict[f'global_lcc_{mask}'],
            'autoconv': pd_dict[f'autoconv_{mask}'],
            'accretn': pd_dict[f'accretn_{mask}'],
            'prect': pd_dict[f'prect_{mask}'],
            'precl': pd_dict[f'precl_{mask}'],
        },
        'pi': {
            'in_cloud_lwp': pi_dict[f'in_cloud_lwp_{mask}'],
            'in_cloud_cdnc': pi_dict[f'in_cloud_cdnc_{mask}'],
            'in_cloud_reff': pi_dict[f'in_cloud_reff_{mask}'],
            'warm_rain_freq': pi_dict[f'warm_rain_freq_{mask}'],
            'precip_amp_freq': pi_dict[f'precip_amp_freq_{mask}'],
            'lcc': pi_dict[f'lcc_{mask}'],
            'global_lcc': pi_dict[f'global_lcc_{mask}'],
            'autoconv': pi_dict[f'autoconv_{mask}'],
            'accretn': pi_dict[f'accretn_{mask}'],
            'prect': pi_dict[f'prect_{mask}'],
            'precl': pi_dict[f'precl_{mask}'],
        },
        'cntl': {
            'in_cloud_lwp': cntl_pd_dict[f'in_cloud_lwp_{mask}'],
            'in_cloud_cdnc': cntl_pd_dict[f'in_cloud_cdnc_{mask}'],
            'in_cloud_reff': cntl_pd_dict[f'in_cloud_reff_{mask}'],
            'warm_rain_freq': cntl_pd_dict[f'warm_rain_freq_{mask}'],
            'precip_amp_freq': cntl_pd_dict[f'precip_amp_freq_{mask}'],
            'lcc': cntl_pd_dict[f'lcc_{mask}'],
            'global_lcc': cntl_pd_dict[f'global_lcc_{mask}'],
            'autoconv': cntl_pd_dict[f'autoconv_{mask}'],
            'accretn': cntl_pd_dict[f'accretn_{mask}'],
            'prect': cntl_pd_dict[f'prect_{mask}'],
            'precl': cntl_pd_dict[f'precl_{mask}'],
        },
        'adjustments': {
            'lwp_adj': regime_adj_dict[f'lwp_adj_{mask}'],
            'cdnc_adj': regime_adj_dict[f'cdnc_adj_{mask}'],
            'reff_adj': regime_adj_dict[f'reff_adj_{mask}'],
            'warm_rain_freq_adj': regime_adj_dict[f'warm_rain_freq_adj_{mask}'],
            'precip_amp_freq_adj': regime_adj_dict[f'precip_amp_freq_adj_{mask}'],
            'lcc_adj': regime_adj_dict[f'lcc_adj_{mask}'],
            'global_lcc_adj': regime_adj_dict[f'global_lcc_adj_{mask}'],
            'autoconv_adj': regime_adj_dict[f'autoconv_adj_{mask}'],
            'accretn_adj': regime_adj_dict[f'accretn_adj_{mask}'],
            'prect_adj': regime_adj_dict[f'prect_adj_{mask}'],
            'precl_adj': regime_adj_dict[f'precl_adj_{mask}'],
        },
        'deltas': {
            'lwp_delta': regime_adj_dict[f'lwp_delta_{mask}'],
            'cdnc_delta': regime_adj_dict[f'cdnc_delta_{mask}'],
            'reff_delta': regime_adj_dict[f'reff_delta_{mask}'],
            'warm_rain_freq_delta': regime_adj_dict[f'warm_rain_freq_delta_{mask}'],
            'precip_amp_freq_delta': regime_adj_dict[f'precip_amp_freq_delta_{mask}'],
            'lcc_delta': regime_adj_dict[f'lcc_delta_{mask}'],
            'global_lcc_delta': regime_adj_dict[f'global_lcc_delta_{mask}'],
            'autoconv_delta': regime_adj_dict[f'autoconv_delta_{mask}'],
            'accretn_delta': regime_adj_dict[f'accretn_delta_{mask}'],
            'prect_delta': regime_adj_dict[f'prect_delta_{mask}'],
            'precl_delta': regime_adj_dict[f'precl_delta_{mask}'],
        },
        'biases': {
            'lwp_bias': regime_adj_dict[f'lwp_bias_{mask}'],
            'cdnc_bias': regime_adj_dict[f'cdnc_bias_{mask}'],
            'reff_bias': regime_adj_dict[f'reff_bias_{mask}'],
            'warm_rain_freq_bias': regime_adj_dict[f'warm_rain_freq_bias_{mask}'],
            'precip_amp_freq_bias': regime_adj_dict[f'precip_amp_freq_bias_{mask}'],
            'lcc_bias': regime_adj_dict[f'lcc_bias_{mask}'],
            'global_lcc_bias': regime_adj_dict[f'global_lcc_bias_{mask}'],
            'autoconv_bias': regime_adj_dict[f'autoconv_bias_{mask}'],
            'accretn_bias': regime_adj_dict[f'accretn_bias_{mask}'],
            'prect_bias': regime_adj_dict[f'prect_bias_{mask}'],
            'precl_bias': regime_adj_dict[f'precl_bias_{mask}'],
        },
    }

    #----------------------------------------------------
    # Add some metadata
    #----------------------------------------------------
    auto_forcing = auto_fac
    accr_forcing = accr_fac


    # Determine whether this is the control configuration
    is_control = (
        auto_forcing == 1.0 and
        accr_forcing == 1.0
    )
    
    # If not control, determine which forcing was changed (basic version)
    if not is_control and (auto_fac != 0.) and (accr_fac != 0.):
        forcing_type = 'combined'
    elif (auto_fac == 0.0) or (accr_fac == 0.0):
        forcing_type = 'denial'
    else:
        forcing_type = 'control'
    
    # Add metadata to case results
    case_results['meta'] = {
        'case_name': case_name,
        'is_control': forcing_type == 'control',
        'forcing_type': forcing_type,
        'auto_forcing': auto_forcing,
        'accr_forcing': accr_forcing,
    }
    print('meta info:',case_results['meta'])

    results[case_name] = case_results
    dumi+=1

    #print(aaaaaa)

case_name: msp1_00
Processing: msp1_00 ; % done: 1.8518518518518516
auto_fac : 1.0
accr_fac : 1.0


/glade/derecho/scratch/mckenna/tmp/ipykernel_51069/1748557754.py:92: RuntimeWarning: divide by zero encountered in divide
  ratio = np.where(valid_lcc_mask, var / lcc, np.nan)  # Normalize by LCC
/glade/derecho/scratch/mckenna/tmp/ipykernel_51069/1748557754.py:92: RuntimeWarning: invalid value encountered in divide
  ratio = np.where(valid_lcc_mask, var / lcc, np.nan)  # Normalize by LCC
/glade/derecho/scratch/mckenna/tmp/ipykernel_51069/1748557754.py:93: RuntimeWarning: Mean of empty slice
  return area_weighted_mean(np.nanmean(ratio, axis=0), area, mask=np.any(valid_lcc_mask, axis=0))
/glade/derecho/scratch/mckenna/tmp/ipykernel_51069/1748557754.py:92: RuntimeWarning: overflow encountered in divide
  ratio = np.where(valid_lcc_mask, var / lcc, np.nan)  # Normalize by LCC
/glade/derecho/scratch/mckenna/tmp/ipykernel_51069/1748557754.py:111: RuntimeWarning: Mean of empty slice
  f'lcc_{label}': area_weighted_mean(np.nanmean(lcc, axis=0), area, mask=np.any(valid_lcc_mask, axis=0)),
/gla


Label: warm,PD
WROF: 0.6183885363089321
PAF: 0.09177302978338882

Label: warm,PI
WROF: 0.6280985084569217
PAF: 0.08931513363013249

Label: warm,CNTL_PD
WROF: 0.6183885363089321
PAF: 0.09177302978338882

Label: warm_overcast,PD
WROF: 0.6183885363089321
PAF: 0.09177302978338882

Label: warm_overcast,PI
WROF: 0.6280985084569217
PAF: 0.08931513363013249

Label: warm_overcast,CNTL_PD
WROF: 0.6183885363089321
PAF: 0.09177302978338882

Label: warm_strat,PD
WROF: 0.6220090564496313
PAF: 0.008270797330246807

Label: warm_strat,PI
WROF: 0.6464306144093949
PAF: 0.007796072225645875

Label: warm_strat,CNTL_PD
WROF: 0.6220090564496313
PAF: 0.008270797330246807

meta info: {'case_name': 'msp1_00', 'is_control': True, 'forcing_type': 'control', 'auto_forcing': 1.0, 'accr_forcing': 1.0}
case_name: msp1_01
Processing: msp1_01 ; % done: 3.7037037037037033
auto_fac : 299.0926451671614
accr_fac : 22.47762310145409

Label: warm,PD
WROF: 0.9018427543458367
PAF: 0.03882146531879217

Label: warm,PI
WROF: 0.9

/glade/derecho/scratch/mckenna/tmp/ipykernel_51069/1748557754.py:222: RuntimeWarning: invalid value encountered in scalar divide
  out['accretn_adj'] = np.log(accretn_pd / accretn_pi)
/glade/derecho/scratch/mckenna/tmp/ipykernel_51069/1748557754.py:317: RuntimeWarning: invalid value encountered in scalar divide
  out[f'accretn_adj_{mask}'] = np.log(accretn_pd / accretn_pi)


meta info: {'case_name': 'msp1_51', 'is_control': False, 'forcing_type': 'denial', 'auto_forcing': 1.0, 'accr_forcing': 0.0}
case_name: msp1_52
Processing: msp1_52 ; % done: 98.14814814814815
auto_fac : 0.0
accr_fac : 1.0

Label: warm,PD
WROF: 0.20748234343841257
PAF: 0.2699601061213526

Label: warm,PI
WROF: 0.20968993290083054
PAF: 0.27462909641240774

Label: warm,CNTL_PD
WROF: 0.6183885363089321
PAF: 0.09177302978338882

Label: warm_overcast,PD
WROF: 0.20748234343841257
PAF: 0.2699601061213526

Label: warm_overcast,PI
WROF: 0.20968993290083054
PAF: 0.27462909641240774

Label: warm_overcast,CNTL_PD
WROF: 0.6183885363089321
PAF: 0.09177302978338882

Label: warm_strat,PD
WROF: 0.0539679967212851
PAF: 0.060249180058933365

Label: warm_strat,PI
WROF: 0.055086766548103766
PAF: 0.06485028157299802

Label: warm_strat,CNTL_PD
WROF: 0.6220090564496313
PAF: 0.008270797330246807



/glade/derecho/scratch/mckenna/tmp/ipykernel_51069/1748557754.py:214: RuntimeWarning: invalid value encountered in scalar divide
  out['autoconv_adj'] = np.log(autoconv_pd / autoconv_pi)
/glade/derecho/scratch/mckenna/tmp/ipykernel_51069/1748557754.py:309: RuntimeWarning: invalid value encountered in scalar divide
  out[f'autoconv_adj_{mask}'] = np.log(autoconv_pd / autoconv_pi)


meta info: {'case_name': 'msp1_52', 'is_control': False, 'forcing_type': 'denial', 'auto_forcing': 0.0, 'accr_forcing': 1.0}
case_name: msp1_53
Processing: msp1_53 ; % done: 100.0
auto_fac : 0.0
accr_fac : 0.0

Label: warm,PD
WROF: 0.20990929820521922
PAF: 0.2819453785792075

Label: warm,PI
WROF: 0.2104690375171628
PAF: 0.2880874885097849

Label: warm,CNTL_PD
WROF: 0.6183885363089321
PAF: 0.09177302978338882

Label: warm_overcast,PD
WROF: 0.20990929820521922
PAF: 0.2819453785792075

Label: warm_overcast,PI
WROF: 0.2104690375171628
PAF: 0.2880874885097849

Label: warm_overcast,CNTL_PD
WROF: 0.6183885363089321
PAF: 0.09177302978338882

Label: warm_strat,PD
WROF: 0.05419479381585494
PAF: 0.06544267552705398

Label: warm_strat,PI
WROF: 0.05475149186906357
PAF: 0.06787813833846729

Label: warm_strat,CNTL_PD
WROF: 0.6220090564496313
PAF: 0.008270797330246807

meta info: {'case_name': 'msp1_53', 'is_control': False, 'forcing_type': 'denial', 'auto_forcing': 0.0, 'accr_forcing': 0.0}


# Save as pickle file

In [23]:
with open(f"/glade/u/home/mckenna/scratch/msp1_all_forcings_v2.pkl", "wb") as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)